# ADJSCC DIV2K → Kodak training (official TensorFlow architecture)

This Kaggle run uses the author repository and TensorFlow Compression 2.14.1 in an isolated Python 3.11 environment, because Kaggle’s base Python 3.13 image does not provide a compatible TensorFlow Compression package. The training model is built from the official ADJSCC `Attention_Encoder`, `Attention_Decoder`, and `Channel`. It trains 150 epochs on each channel condition and saves checkpoints and metrics. Rayleigh uses the official `slow_fading_eq` channel.


In [ ]:
import json,subprocess,sys
from pathlib import Path
!pip -q install uv
!uv python install 3.11
!uv venv /kaggle/working/adjscc311 --python 3.11
!uv pip install --python /kaggle/working/adjscc311/bin/python tensorflow==2.14.1 tensorflow-compression==2.14.1 tensorflow-probability==0.22.1 pillow
!uv pip install --python /kaggle/working/adjscc311/bin/python nvidia-cuda-runtime-cu11==11.8.89 nvidia-cublas-cu11==11.11.3.6 nvidia-cufft-cu11==10.9.0.58 nvidia-cudnn-cu11==8.7.0.84 nvidia-curand-cu11==10.3.0.86 nvidia-cusolver-cu11==11.4.1.48 nvidia-cusparse-cu11==11.7.5.86 nvidia-nccl-cu11==2.16.5 nvidia-cuda-cupti-cu11==11.8.87 nvidia-cuda-nvcc-cu11==11.8.89
!bash -lc 'cd /kaggle/working/adjscc311/lib/python3.11/site-packages/tensorflow && ln -sfn ../nvidia/*/lib/*.so* .'
!bash -lc 'find /kaggle/working/adjscc311/lib/python3.11/site-packages/nvidia/cuda_nvcc -path "*/bin/ptxas" -print -quit | xargs -I{} ln -sfn {} /kaggle/working/adjscc311/bin/ptxas'
PY311='/kaggle/working/adjscc311/bin/python'
import subprocess
gpu_check=subprocess.run([PY311,"-c","import tensorflow as tf; print(tf.__version__,tf.config.list_physical_devices(\"GPU\"))"],capture_output=True,text=True)
print(gpu_check.stdout,gpu_check.stderr,flush=True)
assert gpu_check.returncode==0 and "PhysicalDevice" in gpu_check.stdout, "TensorFlow runtime has no visible GPU"
print("Python 3.11 TFC GPU runtime ready")


In [ ]:
script = "import os,sys,json,csv,time,random,math,subprocess\nfrom pathlib import Path\nimport numpy as np\nfrom PIL import Image\n\nos.environ.setdefault('TF_CPP_MIN_LOG_LEVEL','2')\nimport tensorflow as tf\nfrom tensorflow.keras.layers import Input,Lambda\nfrom tensorflow.keras import Model\nfrom tensorflow.keras.optimizers import Adam\n\nDIV2K=Path('/kaggle/input/notebooks/jagan028/div2k-dataset-generation-for-isr/Training/HR/hr_images')\nKODAK=Path('/kaggle/input/datasets/sherylmehta/kodak-dataset')\nSRC=Path('/kaggle/working/ADJSCC')\nif not SRC.exists(): subprocess.run(['git','clone','https://github.com/alexxu1988/ADJSCC.git',str(SRC)],check=True)\n# The author README identifies this vendored tensorflow_compression directory as macOS-only.\nVENDORED_TFC=SRC/'tensorflow_compression'\nif VENDORED_TFC.exists(): VENDORED_TFC.rename(SRC/'tensorflow_compression_macos_only')\nREV=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'],text=True).strip()\nsys.path.insert(0,str(SRC))\nfrom util_module import Attention_Encoder, Attention_Decoder\nfrom util_channel import Channel\n\nTRAIN_PATHS=sorted(p for p in DIV2K.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg'})\nKODAK_PATHS=sorted(KODAK.glob('kodim*.png'))\nassert len(TRAIN_PATHS)==800,(len(TRAIN_PATHS),DIV2K)\nassert len(KODAK_PATHS)==24,(len(KODAK_PATHS),KODAK)\ngpus=tf.config.list_physical_devices('GPU')\nprint('ADJSCC commit',REV,'DIV2K',len(TRAIN_PATHS),'Kodak',len(KODAK_PATHS),'GPUs',gpus,flush=True)\nassert gpus, 'No Kaggle GPU visible to isolated TensorFlow runtime'\n\nOUT=Path('/kaggle/working/jscc_results/ADJSCC'); OUT.mkdir(parents=True,exist_ok=True)\nEPOCHS=150; BATCH=16; CROP=128; SNR=10.; TCN=16; SEED=20261016\nrandom.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)\nstrategy=tf.distribute.MirroredStrategy()\nprint('TF replicas',strategy.num_replicas_in_sync,flush=True)\n\ndef build(channel):\n    image=Input(shape=(None,None,3),name='image')\n    snr=Input(shape=(1,),name='snr_db')\n    norm=Lambda(lambda z:z/255.,name='normal')(image)\n    encoded=Attention_Encoder(norm,snr,TCN)\n    if channel=='AWGN':\n        z=Channel(channel_type='awgn')(encoded,snr)\n        decoded=Attention_Decoder(z,snr)\n        model=Model([image,snr],Lambda(lambda q:q*255.,name='denormal')(decoded))\n    elif channel=='Rayleigh':\n        hr=Input(shape=(1,),name='h_real'); hi=Input(shape=(1,),name='h_imag')\n        z=Channel(channel_type='slow_fading_eq')(encoded,snr,hr,hi)\n        decoded=Attention_Decoder(z,snr)\n        model=Model([image,snr,hr,hi],Lambda(lambda q:q*255.,name='denormal')(decoded))\n    else: raise ValueError(channel)\n    return model\n\ndef make_data(channel):\n    specs=[tf.TensorSpec((CROP,CROP,3),tf.float32),tf.TensorSpec((1,),tf.float32)]\n    if channel=='Rayleigh': specs += [tf.TensorSpec((1,),tf.float32),tf.TensorSpec((1,),tf.float32)]\n    def gen():\n        rng=np.random.default_rng(SEED)\n        while True:\n            for ix in rng.permutation(len(TRAIN_PATHS)):\n                p=TRAIN_PATHS[int(ix)]\n                a=np.asarray(Image.open(p).convert('RGB'))\n                h,w=a.shape[:2]\n                if h<CROP or w<CROP:\n                    a=np.asarray(Image.fromarray(a).resize((max(w,CROP),max(h,CROP)),Image.Resampling.BICUBIC)); h,w=a.shape[:2]\n                y=int(rng.integers(0,h-CROP+1)); x=int(rng.integers(0,w-CROP+1))\n                crop=a[y:y+CROP,x:x+CROP].astype(np.float32)\n                if rng.random()<.5: crop=crop[:,::-1].copy()\n                example=[crop,np.array([SNR],np.float32)]\n                if channel=='Rayleigh':\n                    example.extend([np.array([rng.normal(0,1/np.sqrt(2))],np.float32),np.array([rng.normal(0,1/np.sqrt(2))],np.float32)])\n                yield tuple(example),crop\n    return tf.data.Dataset.from_generator(gen,output_signature=(tuple(specs),tf.TensorSpec((CROP,CROP,3),tf.float32))).batch(BATCH,drop_remainder=True).prefetch(tf.data.AUTOTUNE)\n\ndef flops_for(concrete):\n    opts=tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()\n    stats=tf.compat.v1.profiler.profile(graph=concrete.graph,options=opts)\n    return int(stats.total_float_ops) if stats is not None else None\n\ndef profile_flops(model,channel):\n    x=tf.zeros((1,CROP,CROP,3),tf.float32); snr=tf.fill((1,1),SNR)\n    hs=[tf.zeros((1,1),tf.float32),tf.ones((1,1),tf.float32)] if channel=='Rayleigh' else []\n    specs=[tf.TensorSpec((1,CROP,CROP,3),tf.float32),tf.TensorSpec((1,1),tf.float32)] + ([tf.TensorSpec((1,1),tf.float32)]*2 if channel=='Rayleigh' else [])\n    @tf.function\n    def infer(*args): return model(list(args),training=False)\n    inference=flops_for(infer.get_concrete_function(*specs))\n    opt=Adam(1e-4)\n    @tf.function\n    def train_step(*args):\n        target=args[0]; inputs=list(args[1:])\n        with tf.GradientTape() as tape:\n            pred=model(inputs,training=True); loss=tf.reduce_mean(tf.square(pred-target))\n        grads=tape.gradient(loss,model.trainable_variables); return loss\n    # Build the traced one-step training graph on a single representative sample.\n    train_specs=[tf.TensorSpec((1,CROP,CROP,3),tf.float32)] + specs\n    training=flops_for(train_step.get_concrete_function(*train_specs))\n    return training,inference\n\nclass EpochCSV(tf.keras.callbacks.Callback):\n    def __init__(self,path,channel,flops): self.path=path; self.channel=channel; self.flops=flops; self.rows=[]\n    def on_epoch_begin(self,epoch,logs=None): self.t0=time.time()\n    def on_epoch_end(self,epoch,logs=None):\n        r={'model':'ADJSCC','channel':self.channel,'epoch':epoch+1,'epochs_target':EPOCHS,'snr_db':SNR,'train_loss_mse_255':float((logs or {}).get('loss',float('nan'))),'epoch_seconds':time.time()-self.t0,'train_flops_per_step':self.flops[0],'inference_flops_per_image':self.flops[1],'tcn':TCN,'nominal_cbr':TCN/96}\n        self.rows.append(r)\n        with self.path.open('w',newline='') as f:\n            w=csv.DictWriter(f,fieldnames=r.keys()); w.writeheader(); w.writerows(self.rows)\n        self.model.save_weights(self.path.parent/'latest.weights.h5')\n        if (epoch+1)%10==0 or epoch==0: print(json.dumps(r),flush=True)\n\ndef eval_kodak(model,channel,run):\n    rows=[]\n    for p in KODAK_PATHS:\n        ref=np.asarray(Image.open(p).convert('RGB')).astype(np.float32)[None,...]\n        inputs=[ref,np.array([[SNR]],np.float32)]\n        if channel=='Rayleigh':\n            rng=np.random.default_rng(SEED+int(p.stem[-2:])); inputs.extend([np.array([[rng.normal(0,1/np.sqrt(2))]],np.float32),np.array([[rng.normal(0,1/np.sqrt(2))]],np.float32)])\n        pred=np.clip(model.predict(inputs,verbose=0),0,255)/255.; target=ref/255.\n        y=tf.convert_to_tensor(target); yh=tf.convert_to_tensor(pred)\n        psnr=float(tf.reduce_mean(tf.image.psnr(yh,y,max_val=1.)).numpy())\n        ss=float(tf.reduce_mean(tf.image.ssim(yh,y,max_val=1.)).numpy())\n        mss=float(tf.reduce_mean(tf.image.ssim_multiscale(yh,y,max_val=1.)).numpy())\n        h,w=ref.shape[1:3]; uses=TCN*(h//4)*(w//4)//2\n        rows.append({'model':'ADJSCC','channel':channel,'snr_db':SNR,'image':p.stem,'psnr_db':psnr,'ssim':ss,'msssim':mss,'real_symbols':TCN*(h//4)*(w//4),'complex_channel_uses':uses,'cbr':uses/(3*h*w)})\n    file=OUT/f'kodak_{channel.lower()}.csv'\n    with file.open('w',newline='') as f:\n        w=csv.DictWriter(f,fieldnames=rows[0].keys()); w.writeheader(); w.writerows(rows)\n    return rows\n\nsummaries=[]\nfor channel in ['AWGN','Rayleigh']:\n    tf.keras.backend.clear_session()\n    with strategy.scope():\n        model=build(channel); model.compile(Adam(1e-4),loss='mse')\n    # Fail before 150 epochs if the official TFC custom SignalConv/GDN path is incompatible.\n    first=next(iter(make_data(channel))); smoke=model(first[0],training=True); assert tuple(smoke.shape)==(BATCH,CROP,CROP,3)\n    flops=profile_flops(model,channel); print('profiled flops',channel,flops,flush=True)\n    run=OUT/channel; run.mkdir(exist_ok=True)\n    cb=EpochCSV(run/'epoch_metrics.csv',channel,flops)\n    model.fit(make_data(channel),epochs=EPOCHS,steps_per_epoch=len(TRAIN_PATHS)//BATCH,callbacks=[cb],verbose=0)\n    rows=eval_kodak(model,channel,run); summary={'model':'ADJSCC','channel':channel,'epochs':EPOCHS,'snr_db':SNR,'kodak_psnr_db':float(np.mean([r['psnr_db'] for r in rows])),'kodak_ssim':float(np.mean([r['ssim'] for r in rows])),'kodak_msssim':float(np.mean([r['msssim'] for r in rows])),'channel_uses_per_image_mean':float(np.mean([r['complex_channel_uses'] for r in rows])),'cbr':float(np.mean([r['cbr'] for r in rows])),'train_flops_per_step':flops[0],'inference_flops_per_image':flops[1],'source_commit':REV}\n    summaries.append(summary); print(json.dumps(summary),flush=True)\n    with (OUT/'summary.csv').open('w',newline='') as f:\n        w=csv.DictWriter(f,fieldnames=summaries[0].keys()); w.writeheader(); w.writerows(summaries)\n    del model; tf.keras.backend.clear_session()\nprint('ADJSCC completed',json.dumps(summaries),flush=True)\n"
Path("/kaggle/working/adjscc_train.py").write_text(script,encoding="utf-8")
subprocess.run([PY311,"/kaggle/working/adjscc_train.py"],check=True)


## Results
Check `/kaggle/working/jscc_results/ADJSCC/summary.csv`, per-image Kodak CSVs, per-epoch metrics, and channel checkpoints. A successful Kaggle job is required before counting the model trained.
